# GRIDTECH v5.5.1 - data-center power portfolio advisor

This notebook builds hourly inputs for any opening month, sizes a portfolio from the seven catalog resources, dispatches all 87,600 hours, applies the common benchmark accounting, ranks options against customer preferences and hard requirements, and exports `portfolio.json` + `dispatch.csv`.

**Run order:** cells 1-7 define the model; section 8 onward runs it. Set `DATA_DIR` in the next cell to the folder holding the packet files (`reference_hourly.csv`, `resource_options.csv`, ...).

Requires `numpy`, `pandas`, `matplotlib` (`tabulate` optional).

## 0 · Setup

In [1]:
from __future__ import annotations
import argparse
import json
import os
import time
from concurrent.futures import ProcessPoolExecutor
from dataclasses import asdict, dataclass, field
from functools import lru_cache

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_DIR = "data"            # <-- folder with the packet CSV/JSON files
OUT_DIR = "outputs"
os.environ["GRIDTECH_DATA"] = DATA_DIR
pd.set_option("display.width", 160)

## 1 · Inputs
Builds the 87,600-hour timeline for any `initial_service_month` and `grid_delay_months` (MODEL_RULES §1-2, §5): commissioning ramp `min(1, 1/3 + k/15)`, staged grid cap 0/25/50/75 MW, 60 % cap in stress hours, and the price / gas / grid-CO2 trajectories. For month 20 with no delay it reproduces `timeline_default.csv`.

In [2]:
HOURS = 87_600
H_MONTH = 730
H_YEAR = 8_760

# Defaults from MODEL_RULES / assumptions_register; overridden by JSON files if present.
DEFAULTS = {
    "wholesale_rate": 0.05,      # price_trajectory.json annual_real_rates.wholesale_energy
    "henry_hub_rate": 0.03,      # price_trajectory.json annual_real_rates.henry_hub
    "grid_co2_rate": -0.03,      # price_trajectory.json annual_real_rates.grid_co2
    "discount_rate": 0.08,       # case.json discount_rate_real (CRF only)
    "land_acres": 8.0,
    "remote_capacity_mw": 12.0,
    "remote_energy_multiplier": 1.05,
    "remote_usd_per_mwh": 70.0,
    "relocation_service_usd": 50.0,
    "interruption_usd": 1000.0,
    "tier2_max_fraction": 1.0,
}


def data_dir(path: str | None = None) -> str:
    """Folder that holds the packet CSV/JSON files."""
    for p in [path, os.environ.get("GRIDTECH_DATA"), "data", "../data", "/mnt/user-data/uploads"]:
        if p and os.path.exists(os.path.join(p, "reference_hourly.csv")):
            return p
    raise FileNotFoundError("Cannot find packet data; set GRIDTECH_DATA to the data folder.")


@lru_cache(maxsize=4)
def load_params(path: str | None = None) -> dict:
    d = data_dir(path)
    p = dict(DEFAULTS)
    try:
        with open(os.path.join(d, "price_trajectory.json")) as f:
            r = json.load(f)["annual_real_rates"]
        p["wholesale_rate"], p["henry_hub_rate"], p["grid_co2_rate"] = (
            r["wholesale_energy"], r["henry_hub"], r["grid_co2"])
    except (FileNotFoundError, KeyError):
        pass
    try:
        with open(os.path.join(d, "response_parameters.json")) as f:
            r = json.load(f)
        for k_json, k in [("remote_capacity_mw", "remote_capacity_mw"),
                          ("remote_energy_multiplier", "remote_energy_multiplier"),
                          ("remote_electricity_usd_per_mwh", "remote_usd_per_mwh"),
                          ("relocation_service_usd_per_local_mwh", "relocation_service_usd"),
                          ("interruption_consequence_usd_per_mwh", "interruption_usd"),
                          ("tier2_max_fraction", "tier2_max_fraction")]:
            if k_json in r:
                p[k] = float(r[k_json])
    except FileNotFoundError:
        pass
    p["relocation_usd"] = p["remote_energy_multiplier"] * p["remote_usd_per_mwh"] + p["relocation_service_usd"]
    p["reloc_cap_local_mw"] = p["remote_capacity_mw"] / p["remote_energy_multiplier"]
    return p


@lru_cache(maxsize=4)
def load_resources(path: str | None = None) -> pd.DataFrame:
    r = pd.read_csv(os.path.join(data_dir(path), "resource_options.csv")).set_index("option_id")
    rate = load_params(path)["discount_rate"]
    crf = rate / (1 - (1 + rate) ** (-r["life_years"]))
    r["annual_usd_per_mw"] = 1000 * (r["capex_usd_per_kw"] * crf + r["fom_usd_per_kw_year"])
    return r


@lru_cache(maxsize=2)
def _reference(path: str | None = None) -> pd.DataFrame:
    return pd.read_csv(os.path.join(data_dir(path), "reference_hourly.csv"))


@dataclass
class Timeline:
    """Hourly arrays (length 87,600) for one opening month and grid delay."""
    opening: int
    delay: int
    month: np.ndarray
    year: np.ndarray
    load: np.ndarray
    t0: np.ndarray
    t1: np.ndarray
    t2: np.ndarray
    cap: np.ndarray          # effective (stress-adjusted) shared import cap
    ordinary_cap: np.ndarray
    stress: np.ndarray       # bool
    wind_cf: np.ndarray
    solar_cf: np.ndarray
    firm_cf: np.ndarray
    price: np.ndarray        # merchant $/MWh incl. adder
    gas: np.ndarray          # delivered $/MMBtu
    co2: np.ndarray          # grid t/MWh (also remote site factor)

    def frame(self) -> pd.DataFrame:
        return pd.DataFrame({k: v for k, v in self.__dict__.items() if isinstance(v, np.ndarray)})


@lru_cache(maxsize=32)
def build_timeline(opening: int = 20, delay: int = 0, path: str | None = None) -> Timeline:
    if not (14 <= opening <= 109):
        raise ValueError("initial_service_month must be an integer in 14..109")
    if delay < 0:
        raise ValueError("grid_delay_months must be >= 0")
    ref = _reference(path)
    p = load_params(path)
    h = np.arange(HOURS)
    month = h // H_MONTH
    rh = h % H_YEAR
    r = {c: ref[c].to_numpy()[rh] for c in ref.columns}

    # commissioning ramp: fraction = min(1, 1/3 + k/15) for k >= 0
    k = month - opening
    frac = np.where(k < 0, 0.0, np.minimum(1.0, 1 / 3 + k / 15))

    # grid schedule (positive milestones shift by delay)
    sched = pd.read_csv(os.path.join(data_dir(path), "grid_connection_schedule.csv"))
    ordinary = np.zeros(HOURS)
    for fm, mw in sched.itertuples(index=False):
        start = fm + delay if mw > 0 else fm
        ordinary = np.where(month >= start, mw, ordinary)
    stress = r["stress_active"].astype(bool)
    cap = ordinary * np.where(stress, r["stress_import_fraction"], 1.0)

    yrs = h / H_YEAR
    price = r["wholesale_price_usd_per_mwh"] * (1 + p["wholesale_rate"]) ** yrs + r["grid_adder_usd_per_mwh"]
    gas = r["henry_hub_usd_per_mmbtu"] * (1 + p["henry_hub_rate"]) ** yrs + r["delivered_gas_adder_usd_per_mmbtu"]
    co2 = r["grid_co2_t_per_mwh"] * (1 + p["grid_co2_rate"]) ** yrs

    return Timeline(opening, delay, month, h // H_YEAR + 1,
                    r["load_mw"] * frac, r["tier0_protected_mw"] * frac,
                    r["tier1_relocatable_mw"] * frac, r["tier2_interruptible_mw"] * frac,
                    cap, ordinary, stress, r["wind_proxy_cf"], r["solar_proxy_cf"],
                    r["firm_clean_available_fraction"].astype(float), price, gas, co2)

## 2 · Dispatch and common accounting
Per hour: (1) contracted wind/solar/nuclear first, up to the shared import cap and load; (2) residual by merit order - fuel cell, gas engine, gas turbine, merchant, Tier 1 relocation (stress only, $123.50/MWh, ≤ 11.43 MW), Tier 2 interruption (stress only, $1,000/MWh or last resort); (3) a chronological battery pass that charges only from otherwise-curtailed clean supply and discharges into each day's five costliest hours. Accounting follows MODEL_RULES §5-6.

In [3]:
GAS = ["fuel_cell", "gas_engine", "gas_turbine"]
CLEAN = ["wind", "utility_solar", "nuclear"]
ALL = ["utility_solar", "wind", "battery_4h", "gas_turbine", "nuclear", "gas_engine", "fuel_cell"]
DISPATCH_COLS = ["model_hour", "grid_mw", "tier1_relocated_mw", "tier2_curtailed_mw",
                 "utility_solar_mw", "wind_mw", "battery_4h_charge_mw", "battery_4h_discharge_mw",
                 "battery_4h_soc_mwh", "gas_turbine_mw", "nuclear_mw", "gas_engine_mw", "fuel_cell_mw"]
LAST_RESORT = 1e6   # cost used for interruption when the customer forbids economic interruption


@dataclass
class Result:
    caps: dict
    opening: int
    delay: int
    feasible: bool
    unmet_mwh: float
    dispatch: pd.DataFrame
    summary: dict = field(default_factory=dict)
    annual: pd.DataFrame | None = None


def validate_caps(caps: dict, path=None) -> list[str]:
    r = load_resources(path)
    land = load_params(path)["land_acres"]
    errs = []
    for k, v in caps.items():
        if k not in r.index:
            errs.append(f"unknown resource {k}")
            continue
        if v < 0 or v > r.at[k, "max_mw"] + 1e-9:
            errs.append(f"{k}={v} outside 0..{r.at[k, 'max_mw']}")
        if abs(v / r.at[k, "block_mw"] - round(v / r.at[k, "block_mw"])) > 1e-9:
            errs.append(f"{k}={v} not a multiple of {r.at[k, 'block_mw']} MW")
    acres = sum(caps.get(k, 0) * r.at[k, "land_acres_per_mw"] for k in r.index)
    if acres > land + 1e-9:
        errs.append(f"land {acres:.2f} acres > {land}")
    return errs


def start_hour(res: str, opening: int, path=None) -> int:
    return max(opening, int(load_resources(path).at[res, "available_month"])) * H_MONTH


def dispatch(tl: Timeline, caps: dict, allow_economic_interrupt: bool = True,
             path=None) -> Result:
    r = load_resources(path)
    p = load_params(path)
    caps = {k: float(caps.get(k, 0)) for k in ALL}
    h = np.arange(HOURS)
    on = {k: (h >= start_hour(k, tl.opening, path)) & (caps[k] > 0) for k in ALL}

    # 1. clean contract availability and dispatch (pro-rata among contracts)
    avail = {"wind": caps["wind"] * tl.wind_cf * on["wind"],
             "utility_solar": caps["utility_solar"] * tl.solar_cf * on["utility_solar"],
             "nuclear": caps["nuclear"] * tl.firm_cf * on["nuclear"]}
    clean_av = sum(avail.values())
    clean_d = np.minimum(clean_av, np.minimum(tl.cap, tl.load))
    resid = tl.load - clean_d
    headroom = tl.cap - clean_d

    # 2. vectorised merit order over the other supply options
    C = np.empty((HOURS, 6))
    Q = np.empty((HOURS, 6))
    for j, g in enumerate(GAS):
        C[:, j] = r.at[g, "payment_usd_per_mwh"] + r.at[g, "heat_rate_mmbtu_per_mwh_hhv"] * tl.gas
        Q[:, j] = caps[g] * on[g]
    C[:, 3], Q[:, 3] = tl.price, headroom
    C[:, 4] = p["relocation_usd"]
    Q[:, 4] = np.where(tl.stress, np.minimum(tl.t1, p["reloc_cap_local_mw"]), 0.0)
    C[:, 5] = p["interruption_usd"] if allow_economic_interrupt else LAST_RESORT
    Q[:, 5] = np.where(tl.stress, tl.t2 * p["tier2_max_fraction"], 0.0)

    order = np.argsort(C, axis=1, kind="stable")
    Qs = np.take_along_axis(Q, order, 1)
    before = np.cumsum(Qs, 1) - Qs
    Ts = np.clip(resid[:, None] - before, 0.0, Qs)
    T = np.zeros_like(Q)
    np.put_along_axis(T, order, Ts, 1)

    # 3. battery pass
    B = caps["battery_4h"]
    charge = np.zeros(HOURS)
    disch = np.zeros(HOURS)
    soc = np.zeros(HOURS)
    if B > 0:
        charge, disch, soc, T, clean_d = _battery(tl, B, on["battery_4h"], C, T, clean_av, clean_d)

    unmet = tl.load - clean_d - T.sum(1) - disch + charge
    unmet_mwh = float(np.clip(unmet, 0, None).sum())
    share = np.divide(clean_d, clean_av, out=np.zeros(HOURS), where=clean_av > 0)

    df = pd.DataFrame({
        "model_hour": h, "grid_mw": T[:, 3], "tier1_relocated_mw": T[:, 4],
        "tier2_curtailed_mw": T[:, 5],
        "utility_solar_mw": avail["utility_solar"] * share, "wind_mw": avail["wind"] * share,
        "battery_4h_charge_mw": charge, "battery_4h_discharge_mw": disch, "battery_4h_soc_mwh": soc,
        "gas_turbine_mw": T[:, 2], "nuclear_mw": avail["nuclear"] * share,
        "gas_engine_mw": T[:, 1], "fuel_cell_mw": T[:, 0]})
    df[DISPATCH_COLS[1:]] = df[DISPATCH_COLS[1:]].clip(lower=0).round(6)
    res = Result(caps, tl.opening, tl.delay, unmet_mwh < 1e-3, unmet_mwh, df)
    res.summary, res.annual = account(tl, caps, df, path)
    return res


def _battery(tl, B, on_b, C, T, clean_av, clean_d):
    """Chronological battery heuristic. Charging uses only curtailed clean supply."""
    eta = np.sqrt(load_resources().at["battery_4h", "round_trip_efficiency"])
    E = 4.0 * B
    n = HOURS
    T = T.copy()
    clean_d = clean_d.copy()
    # marginal cost of the most expensive supply in use each hour
    used = T > 1e-9
    mc = np.where(used.any(1), np.max(np.where(used, C, -np.inf), 1), 0.0)
    # discharge target: hours in the top 5 of each 24-hour window by marginal cost
    # per model day: discharge into the 5 costliest hours (4-hour battery + margin)
    daily = np.sort(mc.reshape(-1, 24), axis=1)[:, -5]
    thr = np.repeat(daily, 24)
    surplus = clean_av - clean_d
    headroom = tl.cap - clean_d - T[:, 3]
    first = int(np.argmax(on_b)) if on_b.any() else n
    end_window = n - 168
    # annual internal-discharge budget, prorated by available hours
    budget = {}
    for y in range(1, 11):
        hrs = on_b[(y - 1) * H_YEAR: y * H_YEAR].sum()
        budget[y] = 365 * E * hrs / H_YEAR
    charge = np.zeros(n)
    disch = np.zeros(n)
    soc = np.zeros(n)
    s_prev = 0.0
    year = tl.year
    for i in range(first, n):
        if i < end_window and surplus[i] > 1e-9 and s_prev < E - 1e-9:
            c = min(B, surplus[i], max(headroom[i], 0.0), (E - s_prev) / eta)
            if c > 1e-9:
                charge[i] = c
                clean_d[i] += c
                s_prev += eta * c
                soc[i] = s_prev
                continue
        final = i >= end_window
        if s_prev > 1e-9 and (final or (mc[i] > 0 and mc[i] >= thr[i])):
            want = min(B, s_prev * eta, budget[year[i]] * eta)
            if want > 1e-9:
                # displace most expensive supply first (clean only in the final window)
                left = want
                for j in np.argsort(-C[i]):
                    if left <= 1e-12:
                        break
                    if not final and C[i, j] < thr[i]:
                        break
                    take = min(T[i, j], left)
                    T[i, j] -= take
                    left -= take
                if final and left > 1e-9:
                    take = min(clean_d[i], left)
                    clean_d[i] -= take
                    left -= take
                d = want - left
                disch[i] = d
                s_prev -= d / eta
                budget[year[i]] -= d / eta
        soc[i] = max(s_prev, 0.0)
    return charge, disch, soc, T, clean_d


def account(tl: Timeline, caps: dict, df: pd.DataFrame, path=None):
    """Common benchmark accounting: cost, CO2, clean share, per model year and horizon."""
    r = load_resources(path)
    p = load_params(path)
    h = df["model_hour"].to_numpy()
    cost_cap = np.zeros(HOURS)
    for k in ALL:
        mw = caps.get(k, 0)
        if mw <= 0:
            continue
        on = h >= start_hour(k, tl.opening, path)
        if r.at[k, "payment_basis"] == "available_generation":
            prof = {"wind": tl.wind_cf, "utility_solar": tl.solar_cf, "nuclear": tl.firm_cf}[k]
            cost_cap += on * mw * prof * r.at[k, "payment_usd_per_mwh"]
        else:
            cost_cap += on * mw * r.at[k, "annual_usd_per_mw"] / H_YEAR
    d = {c: df[c].to_numpy() for c in DISPATCH_COLS}
    gas_cost = sum(d[f"{g}_mw"] * (r.at[g, "payment_usd_per_mwh"] +
                                  r.at[g, "heat_rate_mmbtu_per_mwh_hhv"] * tl.gas) for g in GAS)
    gas_mwh = sum(d[f"{g}_mw"] for g in GAS)
    merchant_cost = d["grid_mw"] * tl.price
    local = cost_cap + gas_cost + merchant_cost
    reloc = d["tier1_relocated_mw"]
    curt = d["tier2_curtailed_mw"]
    reloc_cost = reloc * p["relocation_usd"]
    int_cost = curt * p["interruption_usd"]
    remote_mwh = reloc * p["remote_energy_multiplier"]
    co2_local = d["grid_mw"] * tl.co2 + sum(d[f"{g}_mw"] * r.at[g, "co2_t_per_mwh"] for g in GAS)
    co2_remote = remote_mwh * tl.co2
    clean = (d["wind_mw"] + d["utility_solar_mw"] + d["nuclear_mw"]
             - d["battery_4h_charge_mw"] + d["battery_4h_discharge_mw"])
    served = tl.load - reloc - curt
    cols = {
        "requested_mwh": tl.load, "served_local_mwh": served, "relocated_mwh": reloc,
        "interrupted_mwh": curt, "remote_mwh": remote_mwh, "local_cost_usd": local,
        "relocation_cost_usd": reloc_cost, "interruption_cost_usd": int_cost,
        "co2_local_t": co2_local, "co2_remote_t": co2_remote, "clean_mwh": clean,
        "merchant_mwh": df["grid_mw"].to_numpy(), "gas_mwh": gas_mwh}
    annual = pd.DataFrame({k: np.asarray(v, float).reshape(10, H_YEAR).sum(1) for k, v in cols.items()},
                          index=pd.Index(range(1, 11), name="model_year"))
    tot = annual.sum()

    def derive(x):
        x = x.copy()
        x["total_decision_cost_usd"] = x.local_cost_usd + x.relocation_cost_usd + x.interruption_cost_usd
        x["co2_combined_t"] = x.co2_local_t + x.co2_remote_t
        den = x.served_local_mwh + x.remote_mwh
        x["local_cost_usd_per_mwh"] = x.local_cost_usd / x.served_local_mwh.where(x.served_local_mwh > 0)
        x["decision_cost_usd_per_mwh"] = x.total_decision_cost_usd / x.requested_mwh.where(x.requested_mwh > 0)
        x["combined_kg_co2_per_mwh"] = 1000 * x.co2_combined_t / den.where(den > 0)
        x["clean_share"] = x.clean_mwh / x.served_local_mwh.where(x.served_local_mwh > 0)
        x["merchant_share"] = x.merchant_mwh / x.served_local_mwh.where(x.served_local_mwh > 0)
        return x

    annual = derive(annual)
    summary = derive(pd.DataFrame([tot])).iloc[0].to_dict()
    land = sum(caps.get(k, 0) * r.at[k, "land_acres_per_mw"] for k in ALL)
    summary.update({"initial_service_month": tl.opening, "full_demand_month": tl.opening + 10,
                    "grid_delay_months": tl.delay, "land_acres": land})
    return summary, annual

## 3 · Physical self-check
Independent check of balance, shared cap, tier limits, dates, blocks, land and battery rules. The official `starter/check_submission.py` remains the authority.

In [4]:
TOL = 1e-4


def check(tl: Timeline, caps: dict, df: pd.DataFrame) -> list[str]:
    r = load_resources()
    p = load_params()
    errs = list(validate_caps(caps))
    if list(df.columns) != DISPATCH_COLS or len(df) != HOURS:
        errs.append("dispatch columns/rows do not match the submission schema")
        return errs
    h = df.model_hour.to_numpy()
    if (df[DISPATCH_COLS[1:]] < -TOL).any().any():
        errs.append("negative flow")
    g = lambda c: df[c].to_numpy()
    reloc, curt = g("tier1_relocated_mw"), g("tier2_curtailed_mw")
    gen = sum(g(f"{k}_mw") for k in ["utility_solar", "wind", "nuclear"] + GAS)
    bal = gen + g("grid_mw") + g("battery_4h_discharge_mw") - (tl.load - reloc - curt + g("battery_4h_charge_mw"))
    if np.abs(bal).max() > TOL:
        errs.append(f"energy balance off by {np.abs(bal).max():.4f} MW")
    imports = g("grid_mw") + g("utility_solar_mw") + g("wind_mw") + g("nuclear_mw")
    if (imports - tl.cap).max() > TOL:
        errs.append(f"import cap exceeded by {(imports - tl.cap).max():.3f} MW")
    if (reloc[~tl.stress] > TOL).any() or (curt[~tl.stress] > TOL).any():
        errs.append("relocation/interruption outside stress hours")
    if (reloc - np.minimum(tl.t1, p["reloc_cap_local_mw"])).max() > TOL:
        errs.append("relocation above Tier 1 or remote capacity")
    if (curt - tl.t2 * p["tier2_max_fraction"]).max() > TOL:
        errs.append("interruption above Tier 2 limit")
    prof = {"wind": tl.wind_cf, "utility_solar": tl.solar_cf, "nuclear": tl.firm_cf}
    for k in ALL:
        mw = caps.get(k, 0)
        on = h >= start_hour(k, tl.opening)
        cols = [f"{k}_mw"] if k != "battery_4h" else ["battery_4h_charge_mw", "battery_4h_discharge_mw"]
        for c in cols:
            if (g(c)[~on] > TOL).any():
                errs.append(f"{c} nonzero before commitment start")
            lim = mw * prof[k] if k in prof else mw
            if (g(c) - lim).max() > TOL:
                errs.append(f"{c} above available capacity")
    # battery
    B = caps.get("battery_4h", 0)
    ch, dc, soc = g("battery_4h_charge_mw"), g("battery_4h_discharge_mw"), g("battery_4h_soc_mwh")
    eta = np.sqrt(r.at["battery_4h", "round_trip_efficiency"])
    prev = np.concatenate([[0.0], soc[:-1]])
    if np.abs(prev + eta * ch - dc / eta - soc).max() > 1e-3:
        errs.append("battery SOC equation violated")
    if soc.min() < -1e-3 or soc.max() > 4 * B + 1e-3 or abs(soc[-1]) > 1e-3:
        errs.append("battery SOC bounds or final SOC=0 violated")
    if ((ch > TOL) & (dc > TOL)).any():
        errs.append("simultaneous charge and discharge")
    clean = g("utility_solar_mw") + g("wind_mw") + g("nuclear_mw")
    if (ch - clean).max() > TOL:
        errs.append("battery charged from non-clean supply")
    if B > 0:
        on = h >= start_hour("battery_4h", tl.opening)
        for y in range(1, 11):
            sl = slice((y - 1) * H_YEAR, y * H_YEAR)
            lim = 365 * on[sl].sum() / H_YEAR
            if (dc[sl] / eta).sum() / (4 * B) > lim + 1e-6:
                errs.append(f"battery cycle limit exceeded in year {y}")
    return errs

## 4 · Customer preferences and portfolio search
* **Preferences** (traded off): weights on cost, environment, speed. Score = w_cost·($/MWh ÷ 60) + w_env·(kg CO2/MWh ÷ 300) + w_speed·((full-demand month − 24) ÷ 12).
* **Hard requirements** (never traded off): latest opening / full-demand month, max kg CO2/MWh, max $/MWh, zero Tier 2 interruption.
* **Search:** coordinate descent over catalog blocks from three seeds per opening month, then openings ranked with the speed weight. Good, not provably optimal.

In [5]:
STEPS = {"wind": [10, 30], "utility_solar": [10, 30], "nuclear": [5, 25], "battery_4h": [4, 12],
         "gas_turbine": [25], "gas_engine": [5, 15], "fuel_cell": [5, 15]}
PENALTY = 1e3


@dataclass
class Preferences:
    w_cost: float = 0.5
    w_env: float = 0.3
    w_speed: float = 0.2
    latest_opening: int | None = None        # hard: initial service by this month
    latest_full_demand: int | None = None    # hard: full demand by this month
    max_kg_co2_per_mwh: float | None = None  # hard: combined horizon intensity
    max_cost_usd_per_mwh: float | None = None  # hard: decision cost per requested MWh
    zero_interruption: bool = False          # hard: no Tier 2 interruption at all
    grid_delay_months: int = 0
    openings: list = field(default_factory=lambda: [14, 17, 20, 24, 26])


def _norm(w):
    s = sum(w) or 1.0
    return [x / s for x in w]


def violations(s: dict, prefs: Preferences) -> list[str]:
    v = []
    if prefs.max_kg_co2_per_mwh is not None and s["combined_kg_co2_per_mwh"] > prefs.max_kg_co2_per_mwh:
        v.append(f"CO2 {s['combined_kg_co2_per_mwh']:.0f} > {prefs.max_kg_co2_per_mwh:.0f} kg/MWh")
    if prefs.max_cost_usd_per_mwh is not None and s["decision_cost_usd_per_mwh"] > prefs.max_cost_usd_per_mwh:
        v.append(f"cost {s['decision_cost_usd_per_mwh']:.1f} > {prefs.max_cost_usd_per_mwh:.1f} $/MWh")
    if prefs.zero_interruption and s["interrupted_mwh"] > 1e-6:
        v.append(f"{s['interrupted_mwh']:.0f} MWh Tier 2 interrupted")
    return v


def score(s: dict, prefs: Preferences, with_speed: bool) -> float:
    wc, we, ws = _norm([prefs.w_cost, prefs.w_env, prefs.w_speed])
    val = wc * s["decision_cost_usd_per_mwh"] / 60 + we * s["combined_kg_co2_per_mwh"] / 300
    if with_speed:
        val += ws * (s["full_demand_month"] - 24) / 12
    # graded penalty so the search can walk toward feasibility
    if prefs.max_kg_co2_per_mwh is not None:
        val += PENALTY * max(0, s["combined_kg_co2_per_mwh"] - prefs.max_kg_co2_per_mwh) / 300
    if prefs.max_cost_usd_per_mwh is not None:
        val += PENALTY * max(0, s["decision_cost_usd_per_mwh"] - prefs.max_cost_usd_per_mwh) / 60
    if prefs.zero_interruption:
        val += PENALTY * s["interrupted_mwh"] / 1e4
    return val


class Evaluator:
    def __init__(self, opening, prefs):
        self.tl = build_timeline(opening, prefs.grid_delay_months)
        self.prefs = prefs
        self.cache = {}

    def __call__(self, caps):
        key = tuple(int(round(caps.get(k, 0))) for k in ALL)
        if key not in self.cache:
            if validate_caps(caps):
                self.cache[key] = (float("inf"), None)
            else:
                res = dispatch(self.tl, caps, allow_economic_interrupt=not self.prefs.zero_interruption)
                val = score(res.summary, self.prefs, False) + (PENALTY * (1 + res.unmet_mwh / 1e4) if not res.feasible else 0)
                self.cache[key] = (val, None)   # keep only the score: full dispatches are ~9 MB each
        return self.cache[key]

    def full(self, caps):
        return dispatch(self.tl, caps, allow_economic_interrupt=not self.prefs.zero_interruption)


def seeds(opening):
    """Starting points: enough firm onsite capacity for the commissioning gap."""
    firm = 50 if opening < 18 else 25
    return [
        {"gas_turbine": firm, "wind": 50, "utility_solar": 30},
        {"gas_engine": min(60, firm + 10), "wind": 80, "utility_solar": 50, "battery_4h": 12},
        {"fuel_cell": 30, "gas_engine": max(0, firm - 20), "wind": 100, "utility_solar": 60,
         "battery_4h": 12, "nuclear": 25},
    ]


def optimise(opening, prefs: Preferences, max_evals=120, log=None):
    ev = Evaluator(opening, prefs)
    best_val, best_caps = float("inf"), None
    for seed in seeds(opening):
        caps = {k: seed.get(k, 0) for k in ALL}
        val, _ = ev(caps)
        n0 = len(ev.cache)
        improved = True
        while improved and len(ev.cache) - n0 < max_evals:
            improved = False
            for k in ALL:
                for st in STEPS[k]:
                    for sign in (+1, -1):
                        trial = dict(caps)
                        trial[k] = caps[k] + sign * st
                        if trial[k] < 0:
                            continue
                        tv, _ = ev(trial)
                        if tv < val - 1e-9:
                            caps, val, improved = trial, tv, True
        if val < best_val:
            best_val, best_caps = val, caps
        if log:
            log(f"  opening {opening}: seed done, score {val:.4f}, evals {len(ev.cache)}")
    return ev.full(best_caps), len(ev.cache)


def recommend(prefs: Preferences, log=print, parallel=False):
    """Search each allowed opening month; return (best, table of per-opening results)."""
    openings = [o for o in prefs.openings
                if (prefs.latest_opening is None or o <= prefs.latest_opening)
                and (prefs.latest_full_demand is None or o + 10 <= prefs.latest_full_demand)]
    if not openings:
        raise ValueError("No tested opening month satisfies the timing requirement; relax it or add months.")
    rows = []
    if parallel and len(openings) > 1:
        with ProcessPoolExecutor() as ex:
            outs = list(ex.map(optimise, openings, [prefs] * len(openings)))
    else:
        outs = [optimise(o, prefs, log=log) for o in openings]
    for o, (res, n) in zip(openings, outs):
        if log:
            log(f"opening {o}: searched {n} portfolios")
        s = res.summary
        viol = violations(s, prefs) + ([] if res.feasible else [f"unmet {res.unmet_mwh:.0f} MWh"])
        rows.append({"opening": o, "result": res, "evals": n, "violations": viol,
                     "score": score(s, prefs, True) + (PENALTY if viol else 0)})
    rows.sort(key=lambda x: x["score"])
    best = rows[0]
    return best, rows


def explain(best, rows, prefs: Preferences) -> str:
    s = best["result"].summary
    caps = {k: v for k, v in best["result"].caps.items() if v > 0}
    lines = [f"Recommended: open month {s['initial_service_month']}, full demand month {s['full_demand_month']}",
             f"Portfolio (MW): {caps}  | land {s['land_acres']:.2f}/8 acres"]
    if best["violations"]:
        lines.append("WARNING - no tested option meets all hard requirements: " + "; ".join(best["violations"]))
        lines.append("Suggested relaxation: loosen the binding requirement above, or widen the tested opening months.")
    others = [r for r in rows if r is not best]
    for r in others:
        o = r["result"].summary
        d_cost = o["decision_cost_usd_per_mwh"] - s["decision_cost_usd_per_mwh"]
        d_co2 = o["combined_kg_co2_per_mwh"] - s["combined_kg_co2_per_mwh"]
        tag = "; fails: " + ", ".join(r["violations"]) if r["violations"] else ""
        lines.append(f"  alternative: open m{r['opening']} -> {o['decision_cost_usd_per_mwh']:.2f} $/MWh, "
                     f"{o['combined_kg_co2_per_mwh']:.0f} kg/MWh ({d_cost:+.2f} $/MWh, {d_co2:+.0f} kg vs recommended){tag}")
    wc, we, ws = _norm([prefs.w_cost, prefs.w_env, prefs.w_speed])
    lines.append(f"Weights used (normalised): cost {wc:.2f}, environment {we:.2f}, speed {ws:.2f}")
    return "\n".join(lines)


def prefs_dict(prefs):
    return asdict(prefs)

## 5 · Export helpers

In [6]:
SUMMARY_KEYS = [
    ("initial_service_month", "Initial service month", "{:.0f}"),
    ("full_demand_month", "Full demand month", "{:.0f}"),
    ("requested_mwh", "Requested MWh", "{:,.0f}"),
    ("served_local_mwh", "Served locally MWh", "{:,.0f}"),
    ("relocated_mwh", "Tier 1 relocated MWh", "{:,.0f}"),
    ("interrupted_mwh", "Tier 2 interrupted MWh", "{:,.0f}"),
    ("local_cost_usd", "Local energy cost $", "{:,.0f}"),
    ("relocation_cost_usd", "Relocation cost $", "{:,.0f}"),
    ("interruption_cost_usd", "Interruption consequence $", "{:,.0f}"),
    ("total_decision_cost_usd", "Total decision cost $", "{:,.0f}"),
    ("local_cost_usd_per_mwh", "Local cost $/MWh served", "{:.2f}"),
    ("decision_cost_usd_per_mwh", "Decision cost $/MWh requested", "{:.2f}"),
    ("co2_local_t", "CO2 local t", "{:,.0f}"),
    ("co2_remote_t", "CO2 remote t", "{:,.0f}"),
    ("co2_combined_t", "CO2 combined t", "{:,.0f}"),
    ("combined_kg_co2_per_mwh", "Combined kg CO2/MWh", "{:.1f}"),
    ("clean_share", "Clean share", "{:.1%}"),
    ("merchant_share", "Merchant share of local served", "{:.1%}"),
    ("land_acres", "Onsite land used (of 8 acres)", "{:.2f}"),
]


def portfolio_json(res: Result) -> dict:
    return {"initial_service_month": int(res.opening), "grid_delay_months": int(res.delay),
            "capacities_mw": {k: float(res.caps.get(k, 0)) for k in ALL}}


def write_submission(res: Result, out_dir: str, prefs: dict | None = None, explanation: str = ""):
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "portfolio.json"), "w") as f:
        json.dump(portfolio_json(res), f, indent=2)
    res.dispatch.to_csv(os.path.join(out_dir, "dispatch.csv"), index=False, float_format="%.6f")
    res.annual.to_csv(os.path.join(out_dir, "annual_results.csv"))
    s = res.summary
    lines = ["# GRIDTECH recommendation summary", ""]
    if explanation:
        lines += ["```", explanation, "```", ""]
    lines += ["## Portfolio", "", "| Resource | MW |", "|---|---|"]
    lines += [f"| {k} | {res.caps.get(k, 0):g} |" for k in ALL]
    lines += ["", "## Horizon results (months 0-120)", "", "| Metric | Value |", "|---|---|"]
    lines += [f"| {lab} | {fmt.format(s[k])} |" for k, lab, fmt in SUMMARY_KEYS]
    cols = ["requested_mwh", "relocated_mwh", "interrupted_mwh", "total_decision_cost_usd",
            "decision_cost_usd_per_mwh", "combined_kg_co2_per_mwh", "clean_share"]
    try:
        table = res.annual[cols].round(3).to_markdown()
    except ImportError:          # tabulate not installed
        table = "```\n" + res.annual[cols].round(3).to_string() + "\n```"
    lines += ["", "## Per model year", "", table]
    if prefs:
        lines += ["", "## Customer inputs", "", "```json", json.dumps(prefs, indent=2), "```"]
    with open(os.path.join(out_dir, "results_summary.md"), "w") as f:
        f.write("\n".join(lines))
    stress_plot(res, os.path.join(out_dir, "stress_response.png"))


def stress_plot(res: Result, path: str, year: int = 4):
    """Stacked supply during the stress hours of one mature model year."""
    df = res.dispatch
    tl = build_timeline(res.opening, res.delay)
    m = (tl.year == year) & tl.stress
    d = df[m].reset_index(drop=True)
    x = np.arange(len(d))
    layers = [("wind_mw", "Wind", "#2a78d6"), ("utility_solar_mw", "Solar", "#eda100"),
              ("nuclear_mw", "Nuclear", "#6250d6"), ("grid_mw", "Merchant", "#8c8c8c"),
              ("battery_4h_discharge_mw", "Battery", "#1baf7a"), ("fuel_cell_mw", "Fuel cell", "#e87ba4"),
              ("gas_engine_mw", "Gas engine", "#eb6834"), ("gas_turbine_mw", "Gas turbine", "#e34948"),
              ("tier1_relocated_mw", "Tier 1 relocated", "#9fd8c2"), ("tier2_curtailed_mw", "Tier 2 interrupted", "#f5c97a")]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    bottom = np.zeros(len(d))
    for c, lab, col in layers:
        if d[c].sum() > 1e-6:
            ax.bar(x, d[c], bottom=bottom, color=col, width=0.9, label=lab)
            bottom += d[c].to_numpy()
    ax.step(x, tl.cap[m], where="mid", color="black", lw=1.5, label="Import cap (stress)")
    ax.plot(x, tl.t0[m], color="black", ls="--", lw=1, label="Tier 0 protected")
    ax.set(xlabel=f"Stress hour, model year {year}", ylabel="MW",
           title="Supply and demand response during stress events")
    ax.legend(ncol=4, fontsize=8, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.15))
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)

## 6 · Check the input builder against `timeline_default.csv`

In [7]:
tl20 = build_timeline(20, 0)
ref = pd.read_csv(os.path.join(data_dir(), "timeline_default.csv"))
pairs = {"load": "load_mw", "cap": "effective_grid_limit_mw", "price": "merchant_price_usd_per_mwh",
         "gas": "delivered_gas_price_usd_per_mmbtu", "co2": "grid_co2_t_per_mwh"}
pd.Series({k: np.abs(getattr(tl20, k) - ref[v].to_numpy()).max() for k, v in pairs.items()}, name="max abs diff")

FileNotFoundError: Cannot find packet data; set GRIDTECH_DATA to the data folder.

## 7 · Evaluate one portfolio (quick, no search)

In [ ]:
caps = {"gas_turbine": 25, "wind": 160, "utility_solar": 70}
r = dispatch(build_timeline(20, 0), caps)
print("feasible:", r.feasible, "| self-check:", check(build_timeline(20, 0), caps, r.dispatch) or "PASS")
pd.DataFrame([(lab, fmt.format(r.summary[k])) for k, lab, fmt in SUMMARY_KEYS], columns=["Metric", "Value"])

## 8 · Recommend for a customer
Edit the preferences below. A full search over five opening months takes a few minutes in a notebook (it runs serially here); fewer `openings` is faster.

In [ ]:
prefs = Preferences(
    w_cost=0.5, w_env=0.3, w_speed=0.2,   # preferences (trade-offs)
    latest_opening=None,                  # hard: initial service by month
    latest_full_demand=None,              # hard: full demand by month
    max_kg_co2_per_mwh=None,              # hard: combined intensity cap
    max_cost_usd_per_mwh=None,            # hard: decision cost cap
    zero_interruption=False,              # hard: no Tier 2 interruption
    grid_delay_months=0,
    openings=[14, 17, 20, 24, 26],
)
t0 = time.time()
best, rows = recommend(prefs, log=None)
res = best["result"]
print(explain(best, rows, prefs))
print(f"\n({time.time() - t0:.0f}s)")

In [ ]:
pd.DataFrame([{
    "opening": r["opening"], "full demand": r["opening"] + 10,
    "$/MWh": r["result"].summary["decision_cost_usd_per_mwh"],
    "kg CO2/MWh": r["result"].summary["combined_kg_co2_per_mwh"],
    "clean share": r["result"].summary["clean_share"],
    "requested GWh": r["result"].summary["requested_mwh"] / 1e3,
    "interrupted MWh": r["result"].summary["interrupted_mwh"],
    "meets requirements": not r["violations"],
    "portfolio": {k: v for k, v in r["result"].caps.items() if v},
} for r in sorted(rows, key=lambda r: r["opening"])]).round(2)

### Horizon metrics and per-year results

In [ ]:
display(pd.DataFrame([(lab, fmt.format(res.summary[k])) for k, lab, fmt in SUMMARY_KEYS], columns=["Metric", "Value"]))
res.annual[["requested_mwh", "relocated_mwh", "interrupted_mwh", "total_decision_cost_usd",
            "decision_cost_usd_per_mwh", "combined_kg_co2_per_mwh", "clean_share"]].round(2)

### Behaviour during grid-stress events

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
stress_plot(res, os.path.join(OUT_DIR, "stress_response.png"))
from IPython.display import Image
Image(os.path.join(OUT_DIR, "stress_response.png"))

## 9 · Demonstrate a priority change
Same customer, now with speed first and a hard 100 kg CO2/MWh cap.

In [ ]:
prefs2 = Preferences(w_cost=0.2, w_env=0.3, w_speed=0.5, max_kg_co2_per_mwh=100, openings=[14, 20])
best2, rows2 = recommend(prefs2, log=None)
print(explain(best2, rows2, prefs2))
s1, s2 = res.summary, best2["result"].summary
pd.DataFrame({"base": [s1["full_demand_month"], s1["decision_cost_usd_per_mwh"], s1["combined_kg_co2_per_mwh"], s1["clean_share"], {k: v for k, v in res.caps.items() if v}],
              "speed + CO2 cap": [s2["full_demand_month"], s2["decision_cost_usd_per_mwh"], s2["combined_kg_co2_per_mwh"], s2["clean_share"], {k: v for k, v in best2["result"].caps.items() if v}]},
             index=["full-demand month", "$/MWh", "kg CO2/MWh", "clean share", "portfolio"])

## 10 · Export the submission pair
Then run `python starter/check_submission.py outputs/portfolio.json outputs/dispatch.csv > outputs/check_results.json`.

In [ ]:
errs = check(build_timeline(res.opening, res.delay), res.caps, res.dispatch)
print("Self-check:", errs or "PASS")
write_submission(res, OUT_DIR, prefs_dict(prefs), explain(best, rows, prefs))
print(json.dumps(portfolio_json(res), indent=2))
print(sorted(os.listdir(OUT_DIR)))